In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from surprise import Dataset, Reader, SVD, KNNBasic
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Load Data
df_raw = pd.read_csv("ac-01_telco_customer_behavior_mock_data.csv")

In [ ]:
if df_raw.shape[1] == 1:
    df_telco = df_raw.iloc[:, 0].str.split(',', expand=True)
else:
    df_telco = df_raw.copy()
expected_cols = [
    'customer_id', 'plan_type', 'device_brand', 'avg_data_usage_gb',
    'pct_video_usage', 'avg_call_duration', 'sms_freq', 'monthly_spend',
    'topup_freq', 'travel_score', 'complaint_count', 'target_offer'
]
df_telco.columns = expected_cols
num_cols = [
    'avg_data_usage_gb', 'pct_video_usage', 'avg_call_duration',
    'sms_freq', 'monthly_spend', 'topup_freq',
    'travel_score', 'complaint_count'
]
df_telco[num_cols] = df_telco[num_cols].apply(pd.to_numeric, errors='coerce')
print(f"\n Data Loaded: {df_telco.shape}")

In [ ]:
print(f"\nFirst 5 rows:")
print(df_telco.head())

print(f"\nData Info:")
print(df_telco.info())

print(f"\nMissing Values:")
print(df_telco.isnull().sum())

print(f"\nBasic Statistics:")
print(df_telco.describe())

In [ ]:
# Revenue Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_telco['monthly_spend'], bins=50, edgecolor='black', color='coral')
axes[0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0].set_xlabel('Amount (IDR)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df_telco['monthly_spend'])
axes[1].set_title('Amount Box Plot', fontweight='bold')
axes[1].set_ylabel('Amount (IDR)')

plt.tight_layout()
plt.show()

print(f"\nRevenue Stats:")
print(df_telco['monthly_spend'].describe())

In [ ]:
# User Behavior
user_behavior = df_telco[['customer_id',
                           'monthly_spend',
                           'topup_freq',
                           'avg_data_usage_gb',
                           'avg_call_duration',
                           'sms_freq',
                           'complaint_count']].copy()

user_behavior.rename(columns={
    'monthly_spend': 'monetary',
    'topup_freq': 'frequency'
}, inplace=True)

print(user_behavior.describe())

In [ ]:
# Frequency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_behavior['frequency'], bins=30, edgecolor='black', color='skyblue')
axes[0].set_title('Purchase Frequency', fontweight='bold')
axes[0].set_xlabel('Number of Purchases')
axes[0].set_ylabel('Users')

axes[1].hist(user_behavior['monetary'], bins=30, edgecolor='black', color='lightgreen')
axes[1].set_title('Total Spending', fontweight='bold')
axes[1].set_xlabel('Amount (IDR)')
axes[1].set_ylabel('Users')

plt.tight_layout()
plt.show()

In [ ]:
df_telco.isnull().sum()

categorical_cols = ['plan_type', 'device_brand', 'target_offer']
numeric_cols = [
    'avg_data_usage_gb',
    'pct_video_usage',
    'avg_call_duration',
    'sms_freq',
    'monthly_spend',
    'topup_freq',
    'travel_score',
    'complaint_count'
]

# Untuk kategori → isi dengan 'Unknown'
for col in categorical_cols:
    df_telco[col] = df_telco[col].fillna('Unknown')

# Untuk numerik → isi dengan median
for col in numeric_cols:
    df_telco[col] = df_telco[col].fillna(df_telco[col].median())

print("Missing after cleaning:")
print(df_telco.isnull().sum())

In [ ]:
# Remove Duplicates (Correct Version for Telco Dataset)

print(f"\nDuplicates before: {df_telco.duplicated().sum()}")

df_telco = df_telco.drop_duplicates()

print(f"Duplicates after: {df_telco.duplicated().sum()}")


In [ ]:
# Filter Valid Customers (Correct for Telco Dataset)

df_telco = df_telco[
    (df_telco['monthly_spend'] > 0) &
    (df_telco['topup_freq'] > 0) &
    (df_telco['target_offer'].notna())
]

print(f"\nValid customers after filtering: {len(df_telco):,}")

In [ ]:
# Behavior Feature Engineering

# 1. Average spend per top-up
df_telco['avg_spend_per_topup'] = (
    df_telco['monthly_spend'] / df_telco['topup_freq']
)

# 2. Data intensity (pemakaian data vs belanja)
df_telco['data_intensity'] = (
    df_telco['avg_data_usage_gb'] / df_telco['monthly_spend']
)

# 3. Communication intensity (telpon + SMS)
df_telco['communication_intensity'] = (
    df_telco['avg_call_duration'] + df_telco['sms_freq']
)

# 4. Risk score sederhana
df_telco['risk_score'] = (
    df_telco['complaint_count'] * 0.7 + (1 - df_telco['travel_score']) * 0.3
)

print("New features added:")
print(df_telco[['avg_spend_per_topup',
                 'data_intensity',
                 'communication_intensity',
                 'risk_score']].head())

In [ ]:
# Save cleaned dataset
df_telco.to_csv('telco_customers_cleaned.csv', index=False)
print("\n Saved: telco_customers_cleaned.csv")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)

print(f"Total Customers: {len(df_telco):,}")
print(f"Unique Customers: {df_telco['customer_id'].nunique():,}")
print(f"Unique Product Offers: {df_telco['target_offer'].nunique():,}")
print(f"Total Monthly Revenue: Rp {df_telco['monthly_spend'].sum():,.0f}")

## **2. Feature Engineering**

In [ ]:
# Load cleaned telco data
df_telco = pd.read_csv('telco_customers_cleaned.csv')

print(df_telco.shape)
df_telco.head()

In [ ]:
# RFM Features
print("STEP 1: BEHAVIOR-BASED RFM FEATURES")

rfm = df_telco[['customer_id', 'topup_freq', 'monthly_spend', 'complaint_count']].copy()

# Monetary
rfm['monetary'] = rfm['monthly_spend']

# Frequency
rfm['frequency'] = rfm['topup_freq']

# Recency Proxy (semakin kecil complaint → semakin aktif)
rfm['recency'] = 1 / (rfm['complaint_count'] + 1)

# Ambil hanya kolom final RFM
rfm = rfm[['customer_id', 'recency', 'frequency', 'monetary']]

print(rfm.head())
print(f"\n RFM Stats:\n{rfm.describe()}")

In [ ]:
# Visualize Behavior-Based RFM

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Recency (proxy dari complaint_count)
axes[0].hist(rfm['recency'], bins=50, edgecolor='black')
axes[0].set_title('Recency (Complaint-Based Activity)', fontweight='bold')
axes[0].set_xlabel('Recency Score (Higher = More Active)')
axes[0].set_ylabel('Number of Customers')

# Frequency (topup_freq)
axes[1].hist(rfm['frequency'], bins=30, edgecolor='black')
axes[1].set_title('Frequency (Top-up Frequency)', fontweight='bold')
axes[1].set_xlabel('Top-up Frequency')
axes[1].set_ylabel('Number of Customers')

# Monetary (monthly_spend)
axes[2].hist(rfm['monetary'], bins=50, edgecolor='black')
axes[2].set_title('Monetary (Monthly Spend)', fontweight='bold')
axes[2].set_xlabel('Monthly Spend (IDR)')
axes[2].set_ylabel('Number of Customers')

plt.tight_layout()
plt.show()


In [ ]:
# RFM Scorring
print("STEP 2: BEHAVIOR-BASED RFM SCORING")

# R Score
# recency kita = 1 / (complaint + 1)
# nilai makin besar = makin baik → skor makin besar = makin baik
rfm['r_score'] = pd.qcut(
    rfm['recency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# F Score (topup frequency)
rfm['f_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# M Score (monthly spend)
rfm['m_score'] = pd.qcut(
    rfm['monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# Combined RFM score
rfm['rfm_score'] = (
    rfm['r_score'].astype(str) +
    rfm['f_score'].astype(str) +
    rfm['m_score'].astype(str)
)

print("RFM Score Examples:")
print(rfm[['customer_id', 'r_score', 'f_score', 'm_score', 'rfm_score']].head(10))


In [ ]:
print("STEP 3: ARPU CALCULATION")

# Jika ingin pakai asumsi 3 bulan
MONTHS = 3

# Hitung ARPU (Average Revenue Per User)
arpu = df_telco[['customer_id', 'monthly_spend']].copy()
arpu['arpu'] = arpu['monthly_spend']  # dataset ini sudah monthly

# Opsional: jika ingin rata-rata 3 bulan
# arpu['arpu'] = arpu['monthly_spend'] * 3 / MONTHS

# ARPU Buckets
arpu['arpu_bucket'] = pd.cut(
    arpu['arpu'],
    bins=[0, 50000, 100000, 200000, float('inf')],
    labels=['low', 'medium', 'high', 'premium']
)

print(arpu.head())
print("\nARPU Bucket Distribution:")
print(arpu['arpu_bucket'].value_counts())

In [ ]:
# Visualize ARPU
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram ARPU
axes[0].hist(arpu['arpu'], bins=50, edgecolor='black', color='teal')
axes[0].set_title('ARPU Distribution', fontweight='bold')
axes[0].set_xlabel('ARPU (IDR)')
axes[0].set_ylabel('Number of Customers')

counts = arpu['arpu_bucket'].value_counts().sort_index()
for i, val in enumerate(counts):
    axes[1].text(i, val + 10, f'{val}', ha='center')

# Bar chart ARPU buckets
arpu['arpu_bucket'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='purple', edgecolor='black'
)
axes[1].set_title('ARPU Buckets', fontweight='bold')
axes[1].set_xlabel('Bucket')
axes[1].set_ylabel('Number of Customers')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Usage Features
print("STEP 4: USAGE FEATURES")

np.random.seed(42)
usage_7d = pd.DataFrame({
    'customer_id': df_telco['customer_id'],
    'usage_7d_data_mb': (df_telco['avg_data_usage_gb'] * 1024 / 4).astype(int),
    'usage_7d_voice_min': (df_telco['avg_call_duration'] * 7).astype(int),
    'usage_7d_sms': (df_telco['sms_freq'] / 4).astype(int)
})

print(usage_7d.head())

In [ ]:
# Churn Score
print("STEP 5: CHURN SCORE")

# Normalisasi fitur agar skala 0–1
df_telco['data_norm'] = df_telco['avg_data_usage_gb'] / df_telco['avg_data_usage_gb'].max()
df_telco['call_norm'] = df_telco['avg_call_duration'] / df_telco['avg_call_duration'].max()
df_telco['spend_norm'] = df_telco['monthly_spend'] / df_telco['monthly_spend'].max()
df_telco['topup_norm'] = df_telco['topup_freq'] / df_telco['topup_freq'].max()
df_telco['complaint_norm'] = df_telco['complaint_count'] / df_telco['complaint_count'].max()

# Churn score (0 = sangat loyal, 1 = sangat berisiko)
df_telco['churn_score'] = (
    (1 - df_telco['data_norm']) * 0.25 +
    (1 - df_telco['call_norm']) * 0.20 +
    (1 - df_telco['spend_norm']) * 0.25 +
    (1 - df_telco['topup_norm']) * 0.15 +
    (df_telco['complaint_norm']) * 0.15
)

print(df_telco[['customer_id', 'churn_score']].head())

In [ ]:
rfm = rfm.merge(
    df_telco[['customer_id', 'churn_score']],
    on='customer_id',
    how='left'
)

print(rfm.head())

In [ ]:
# Merge Features
print("STEP 6: MERGE FEATURES")

# Merge RFM + ARPU
user_features = rfm.merge(
    arpu[['customer_id', 'arpu', 'arpu_bucket']],
    on='customer_id',
    how='left'
)

# Merge dengan Usage 7D
user_features = user_features.merge(
    usage_7d,
    on='customer_id',
    how='left'
)

# 1. Isi kolom numerik dengan 0
num_cols = user_features.select_dtypes(include=['int64', 'float64']).columns
user_features[num_cols] = user_features[num_cols].fillna(0)

# 2. Isi kolom kategorikal dengan 'Unknown'
cat_cols = user_features.select_dtypes(include=['category', 'object']).columns

for col in cat_cols:
    if user_features[col].dtype.name == 'category':
        user_features[col] = user_features[col].cat.add_categories(['Unknown'])
    user_features[col] = user_features[col].fillna('Unknown')


print(f"\nFinal Features Shape: {user_features.shape}")
print(f"Columns: {user_features.columns.tolist()}")
print(user_features.head())

In [ ]:
# Save Feature Engineering
output_file = "user_features.csv"
user_features.to_csv(output_file, index=False)

print("\n File berhasil disimpan!")
print(f" Nama File  : {output_file}")
print(f" Total User : {len(user_features):,}")
print(f" Total Fitur: {len(user_features.columns)}")

# FEATURE ENGINEERING SUMMARY

print("FEATURE ENGINEERING COMPLETE")

print("\n Daftar Fitur:")
for col in user_features.columns:
    print(f"- {col}")

print("\n Dataset siap digunakan untuk modeling (Clustering / ML).")

### **SEGMENTATION (K=7 clusters)**

In [ ]:
# Load features
user_features = pd.read_csv('user_features.csv')

print("K-MEANS CLUSTERING")

print("FEATURE SELECTION")
print("-"*60)

feature_cols = [
    'recency',
    'frequency',
    'monetary',
    'arpu',
    'usage_7d_data_mb',
    'churn_score'
]

# Cek apakah semua fitur tersedia
missing_cols = [col for col in feature_cols if col not in user_features.columns]
if missing_cols:
    raise ValueError(f"Kolom ini belum ada di dataset: {missing_cols}")

X = user_features[feature_cols].copy()

# Handle missing values sebelum transformasi
X = X.fillna(0)

# LOG TRANSFORMATION (untuk skewed data)
X['log_monetary'] = np.log1p(X['monetary'])
X['log_arpu'] = np.log1p(X['arpu'])

clustering_features = [
    'recency',
    'frequency',
    'log_monetary',
    'log_arpu',
    'usage_7d_data_mb',
    'churn_score'
]

X_cluster = X[clustering_features]

print(f" Features used for clustering:")
print(clustering_features)

print("\n Statistical Summary:")
print(X_cluster.describe())

In [ ]:
print("FEATURE SCALING")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

X_scaled_df = pd.DataFrame(X_scaled, columns=X_cluster.columns)

print(f" Scaled shape: {X_scaled_df.shape}")
print("\nScaled Feature Preview:")
print(X_scaled_df.head())

In [ ]:
# PCA
print("PCA FOR DIMENSIONALITY REDUCTION")

pca = PCA(n_components=0.9)
X_pca = pca.fit_transform(X_scaled)

print("Original dimension:", X_scaled.shape[1])
print("Reduced dimension after PCA:", X_pca.shape[1])

In [ ]:
print("ELBOW METHOD")

wcss = []
sil_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=30)
    labels = kmeans.fit_predict(X_pca)

    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_pca, labels))

    print(f"K={k}: Inertia={kmeans.inertia_:.0f}, Silhouette={silhouette_score(X_pca, labels):.4f}")

# Plot Elbow & Silhouette
plt.figure(figsize=(14, 5))

# Elbow Plot
plt.subplot(1, 2, 1)
plt.plot(k_range, wcss, marker='o', linewidth=2)
plt.title('Elbow Method', fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.grid(alpha=0.3)

# Silhouette Plot
plt.subplot(1, 2, 2)
plt.plot(k_range, sil_scores, marker='o', linewidth=2)
plt.title('Silhouette Score')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Score')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Pakai K terbaik dari Silhouette
print("TRAIN FINAL K-MEANS")

best_k = sil_scores.index(max(sil_scores)) + 2
print("Best K from Silhouette:", best_k)

kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=30,
    max_iter=300
)

# Training & Assign Cluster (PAKAI X_pca)
user_features['segment_id'] = kmeans_final.fit_predict(X_pca)

# Evaluation Metrics (HARUS pakai X_pca juga)
silhouette = silhouette_score(X_pca, user_features['segment_id'])
davies_bouldin = davies_bouldin_score(X_pca, user_features['segment_id'])

print(f"K = {best_k}")
print(f"Silhouette Score       : {silhouette:.4f}")
print(f"Davies-Bouldin Index   : {davies_bouldin:.4f}")
print(f"Inertia                : {kmeans_final.inertia_:.0f}")

print(f"\nSegment Distribution:")
print(user_features['segment_id'].value_counts().sort_index())

In [ ]:
# Ensure log_monetary and log_arpu are in user_features for profiling
if 'log_monetary' not in user_features.columns:
    user_features['log_monetary'] = np.log1p(user_features['monetary'])
if 'log_arpu' not in user_features.columns:
    user_features['log_arpu'] = np.log1p(user_features['arpu'])

segment_profile = user_features.groupby('segment_id')[clustering_features].mean()
segment_profile

In [ ]:
# Profil numerik tiap segmen
print("CLUSTER PROFILING")

profiles = (
    user_features
    .groupby('segment_id')[clustering_features]
    .mean()
    .round(3)
)

print(profiles)

# Interpretasi otomatis
for seg_id in sorted(user_features['segment_id'].unique()):
    profile = user_features[user_features['segment_id'] == seg_id]

    avg_freq = profile['frequency'].mean()
    avg_monetary = profile['monetary'].mean()
    avg_recency = profile['recency'].mean()
    avg_usage = profile['usage_7d_data_mb'].mean()
    avg_churn = profile['churn_score'].mean()

    if avg_usage > 2000 and avg_churn < 0.65:
        label = "High Value Active Users"
    elif avg_churn >= 0.65:
        label = "Churn Risk Users"
    else:
        label = "Medium Users"

    print(f"\nSegment {seg_id}: {label}")
    print(f"  Size: {len(profile)} ({len(profile)/len(user_features)*100:.1f}%)")
    print(f"  Avg Frequency: {avg_freq:.2f}")
    print(f"  Avg Usage 7D  : {avg_usage:.0f} MB")
    print(f"  Avg Monetary : Rp {avg_monetary:,.0f}")
    print(f"  Avg Recency  : {avg_recency:.2f}")
    print(f"  Avg Churn    : {avg_churn:.3f}")

Hasil segmentasi pelanggan menggunakan metode K-Means dengan jumlah klaster optimal K = 2 menghasilkan dua segmen utama. Segmen 0 merepresentasikan pelanggan dengan tingkat penggunaan data, ARPU, dan nilai transaksi yang tinggi serta risiko churn yang lebih rendah, sehingga dikategorikan sebagai pelanggan bernilai tinggi (high value users). Sementara itu, Segmen 1 menunjukkan karakteristik penggunaan dan pendapatan yang lebih rendah dengan skor churn yang lebih tinggi, sehingga dikategorikan sebagai pelanggan berisiko churn. Segmentasi ini dapat dimanfaatkan untuk perancangan strategi retensi, promosi, dan peningkatan pendapatan pelanggan secara lebih terarah.

In [ ]:
# Segment distribusion (BAR)
print("VISUALIZATION")

plt.figure(figsize=(10, 6))

segment_counts = user_features['segment_id'].value_counts().sort_index()

bars = plt.bar(
    segment_counts.index,
    segment_counts.values,
    edgecolor='black'
)

plt.title('Segment Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Segment')
plt.ylabel('Users')
plt.xticks(segment_counts.index, [f'Seg {i}' for i in segment_counts.index])

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{int(height):,}',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap profiling
plt.figure(figsize=(10, 6))

sns.heatmap(
    profiles.T,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5
)

plt.title('Segment Characteristics', fontsize=14, fontweight='bold')
plt.xlabel('Segment')
plt.ylabel('Feature')

plt.tight_layout()
plt.show()

In [ ]:
# Save user feature result
print("SAVE MODEL & DATA")

user_features.to_csv('user_features_segmented.csv', index=False)
print(" Saved: user_features_segmented.csv")

# Save model & scaller
import joblib

joblib.dump(
    {
        'model': kmeans_final,
        'scaler': scaler,
        'pca': pca
    },
    'kmeans_model.pkl'
)

# Final status
print("SEGMENTATION COMPLETE ")
print(f"Total Users: {len(user_features):,}")
print(f"Total Segments: {user_features['segment_id'].nunique()}")

In [ ]:
# Load data & Merge
user_features = pd.read_csv('user_features_segmented.csv')
transactions  = pd.read_csv('telco_customers_cleaned.csv')

labels = transactions[['customer_id','target_offer']].drop_duplicates('customer_id')
df = user_features.merge(labels, on='customer_id', how='left')
df = df.dropna(subset=['target_offer'])

print("Dataset shape:", df.shape)
print("Unique offers:", df['target_offer'].nunique())
df[['customer_id','target_offer']].head()

In [ ]:
# Filter rare label
label_dist = df['target_offer'].value_counts()
rare_labels = label_dist[label_dist < 30].index
df = df[~df['target_offer'].isin(rare_labels)].reset_index(drop=True)

print("After rare filter:", df.shape)

In [ ]:
# Pilih fitur numerik/categorical yang relevan (sesuaikan)
feature_cols = [
    'recency','frequency','monetary','arpu','usage_7d_data_mb',
    'usage_7d_voice_min','usage_7d_sms','churn_score','avg_spend_per_topup',
    'data_intensity','communication_intensity','risk_score'
]
# Pastikan kolom ada
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].fillna(0)
y = df['target_offer'].astype(str)

# Encode label untuk multiclass
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [ ]:
# Train Test split dataset
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y_enc, df['customer_id'],
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test)

In [ ]:
# Class weight imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

# Initialize params_tuned if it hasn't been defined yet
if 'params_tuned' not in locals():
    params_tuned = {}

params_tuned['class_weight'] = class_weight_dict

In [ ]:
# Train model tuned version
params_tuned = {
    'objective': 'multiclass',
    'num_class': len(le.classes_),
    'metric': 'multi_logloss',
    'learning_rate': 0.03,
    'num_leaves': 16,
    'max_depth': 6,
    'min_data_in_leaf': 80,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 5,
    'lambda_l1': 1.0,
    'lambda_l2': 2.0,
    'seed': 42,
    'verbose': -1
}

model = lgb.train(
    params_tuned,
    train_data,
    valid_sets=[valid_data],
    num_boost_round=500,
    callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
)

print("Model trained")

In [ ]:
# Top-K prediction
proba = model.predict(X_test)

def temperature_scaling(proba, T=0.8):
    logits = np.log(np.clip(proba, 1e-9, 1))
    scaled_logits = logits / T
    exp_logits = np.exp(scaled_logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

proba_scaled = temperature_scaling(proba, T=0.8)

def topk_threshold_safe(proba_row, k=3, min_prob=0.25):
    idxs_sorted = np.argsort(proba_row)[::-1]
    preds = []

    for i in idxs_sorted:
        if proba_row[i] >= min_prob:
            preds.append(le.classes_[i])
        if len(preds) == k:
            break

    if len(preds) < k:
        for i in idxs_sorted:
            label = le.classes_[i]
            if label not in preds:
                preds.append(label)
            if len(preds) == k:
                break

    return preds

topk_preds = [topk_threshold_safe(p, k=3, min_prob=0.25) for p in proba_scaled]

In [ ]:
# Result
res_df = pd.DataFrame({
    'customer_id': ids_test.values,
    'true_label': le.inverse_transform(y_test),
    'pred_top3': topk_preds
})

res_df.head(10)

In [ ]:
# Evaluation
def precision_recall_at_k_preds(true, pred_list, k=3):
    precisions, recalls = [], []
    for t, p in zip(true, pred_list):
        tset, pset = {t}, set(p[:k])
        hit = len(tset & pset)
        precisions.append(hit / k)
        recalls.append(hit)
    return np.mean(precisions), np.mean(recalls)

prec_tuned, rec_tuned = precision_recall_at_k_preds(
    res_df['true_label'].values,
    res_df['pred_top3'].values, # Corrected column name here
    k=3
)

print("TUNED Precision@3:", round(prec_tuned, 4))
print("TUNED Recall@3:", round(rec_tuned, 4))

In [ ]:
# Popularity by segment
train_df_for_popularity = pd.DataFrame({
    'customer_id': ids_train.values,
    'target_offer': le.inverse_transform(y_train)
})

popular_by_segment = (
    train_df_for_popularity
    .merge(user_features[['customer_id','segment_id']], on='customer_id', how='left')
    .groupby(['segment_id','target_offer'])
    .size()
    .reset_index(name='count')
)

popular_by_segment['pop_score'] = (
    popular_by_segment
    .groupby('segment_id')['count']
    .transform(lambda x: x / x.max())
)

In [ ]:
# Global popularity for fallback
popular_global = (
    train_df_for_popularity
    .groupby('target_offer')
    .size()
    .reset_index(name='count')
)

In [ ]:
# Feature importance & analysis (rapi df)
imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importance()
}).sort_values('importance', ascending=False)

imp.head(20)

# End of Code